In [1]:
class ATRStrategy:
    """
    ATR-based strategy for active/swing trades.

    This strategy uses ATR (Average True Range) to determine
    a dynamic stop-loss level.

    Stop Loss:
        Entry Price - (ATR × multiplier)

    This class DOES NOT execute real trades.
    It only generates trading decisions and manages
    the stop-loss information for an active trade.
    """

    def __init__(
        self,
        atr_multiplier=1.5,
        enabled=True
    ):
        """
        Initialize the ATR strategy.

        Parameters
        ----------
        atr_multiplier : float
            Multiplier applied to ATR when calculating
            the stop-loss.

        enabled : bool
            Whether the ATR strategy is enabled.
        """

        self.atr_multiplier = float(atr_multiplier)
        self.enabled = enabled

        # Information about the current active trade
        self.entry_price = None
        self.entry_atr = None
        self.stop_price = None

    def calculate_stop_loss(self, entry_price, atr):
        """
        Calculate the stop-loss price.

        Formula:
            Stop = Entry Price - (ATR × multiplier)

        Parameters
        ----------
        entry_price : float
            Price at which the trade was entered.

        atr : float
            Current ATR value.

        Returns
        -------
        float
            Stop-loss price.
        """

        entry_price = float(entry_price)
        atr = float(atr)

        if entry_price <= 0:
            raise ValueError("Entry price must be greater than zero.")

        if atr < 0:
            raise ValueError("ATR cannot be negative.")

        stop_price = (
            entry_price
            - (self.atr_multiplier * atr)
        )

        return stop_price

    def generate_entry_signal(
        self,
        current_price,
        atr,
        rsi=None,
        macd=None,
        macd_signal=None,
        volume_ratio=None
    ):
        """
        Generate a potential swing-trade entry signal.

        This is intentionally conservative.

        The strategy looks for optional confirmation from:
            - RSI
            - MACD
            - Volume

        Parameters
        ----------
        current_price : float
            Current BTC price.

        atr : float
            Current ATR(14).

        rsi : float, optional
            RSI value.

        macd : float, optional
            MACD value.

        macd_signal : float, optional
            MACD signal value.

        volume_ratio : float, optional
            Current volume divided by average volume.

        Returns
        -------
        dict
            Entry decision.
        """

        current_price = float(current_price)
        atr = float(atr)

        if not self.enabled:
            return {
                "action": "HOLD",
                "price": current_price,
                "reason": "ATR strategy is disabled"
            }

        if current_price <= 0:
            raise ValueError("Current price must be greater than zero.")

        if atr <= 0:
            return {
                "action": "HOLD",
                "price": current_price,
                "reason": "Invalid or unavailable ATR"
            }

        # -------------------------------------------------
        # Confirmation conditions
        # -------------------------------------------------

        confirmations = 0
        reasons = []

        # RSI confirmation
        if rsi is not None:
            if 40 <= rsi <= 70:
                confirmations += 1
                reasons.append("RSI supports momentum")

        # MACD confirmation
        if macd is not None and macd_signal is not None:
            if macd > macd_signal:
                confirmations += 1
                reasons.append("MACD bullish")

        # Volume confirmation
        if volume_ratio is not None:
            if volume_ratio >= 1.2:
                confirmations += 1
                reasons.append("volume breakout")

        # -------------------------------------------------
        # Entry rule
        #
        # Require at least two confirmations when
        # indicators are available.
        # -------------------------------------------------

        available_confirmations = sum([
            rsi is not None,
            macd is not None and macd_signal is not None,
            volume_ratio is not None
        ])

        if available_confirmations >= 2 and confirmations >= 2:

            stop_price = self.calculate_stop_loss(
                current_price,
                atr
            )

            return {
                "action": "BUY",
                "price": current_price,
                "atr": atr,
                "stop_price": stop_price,
                "atr_multiplier": self.atr_multiplier,
                "confirmations": confirmations,
                "reason": "; ".join(reasons)
            }

        return {
            "action": "HOLD",
            "price": current_price,
            "atr": atr,
            "confirmations": confirmations,
            "reason": "Insufficient bullish confirmations"
        }

    def open_trade(self, entry_price, atr):
        """
        Record an active trade.

        This does not execute a Coinbase order.
        """

        self.entry_price = float(entry_price)
        self.entry_atr = float(atr)

        self.stop_price = self.calculate_stop_loss(
            self.entry_price,
            self.entry_atr
        )

    def check_stop_loss(self, current_price):
        """
        Check whether the active trade has reached
        its stop-loss.

        Returns
        -------
        dict
            Stop-loss decision.
        """

        current_price = float(current_price)

        if self.entry_price is None:
            return {
                "action": "NO_TRADE",
                "price": current_price,
                "reason": "No active trade"
            }

        if current_price <= self.stop_price:

            loss_pct = (
                (current_price - self.entry_price)
                / self.entry_price
            ) * 100

            return {
                "action": "SELL",
                "price": current_price,
                "entry_price": self.entry_price,
                "stop_price": self.stop_price,
                "loss_pct": loss_pct,
                "reason": "ATR stop-loss triggered"
            }

        return {
            "action": "HOLD",
            "price": current_price,
            "entry_price": self.entry_price,
            "stop_price": self.stop_price,
            "reason": "Stop-loss not triggered"
        }

    def close_trade(self):
        """
        Clear the active trade information.
        """

        self.entry_price = None
        self.entry_atr = None
        self.stop_price = None

print("atr_strategy.py created successfully!")

atr_strategy.py created successfully!


In [2]:
atr_strategy = ATRStrategy(
    atr_multiplier=1.5,
    enabled=True
)

In [3]:
entry_price = 62500
atr = 1000

stop_price = atr_strategy.calculate_stop_loss(
    entry_price,
    atr
)

print("Entry price:", entry_price)
print("ATR:", atr)
print("ATR multiplier:", atr_strategy.atr_multiplier)
print("Stop-loss:", stop_price)

Entry price: 62500
ATR: 1000
ATR multiplier: 1.5
Stop-loss: 61000.0


In [4]:
atr_strategy.open_trade(
    entry_price=62500,
    atr=1000
)

In [5]:
print("Entry:", atr_strategy.entry_price)
print("Stop:", atr_strategy.stop_price)

Entry: 62500.0
Stop: 61000.0


In [6]:
result = atr_strategy.check_stop_loss(62000)

print(result)

{'action': 'HOLD', 'price': 62000.0, 'entry_price': 62500.0, 'stop_price': 61000.0, 'reason': 'Stop-loss not triggered'}


In [7]:
result = atr_strategy.check_stop_loss(60900)

print(result)

{'action': 'SELL', 'price': 60900.0, 'entry_price': 62500.0, 'stop_price': 61000.0, 'loss_pct': -2.56, 'reason': 'ATR stop-loss triggered'}


In [8]:
import pandas as pd
df_indicators = pd.read_csv("../../data/btc_indicators.csv")
latest = df_indicators.iloc[-1]

current_price = latest["close"]
atr = latest["atr_14"]
rsi = latest["rsi_14"]
macd = latest["macd"]
macd_signal = latest["macd_signal"]
volume_ratio = latest["volume_ratio"]

print("BTC:", current_price)
print("ATR:", atr)
print("RSI:", rsi)
print("MACD:", macd)
print("MACD Signal:", macd_signal)
print("Volume Ratio:", volume_ratio)

BTC: 77524.59
ATR: 352.51520307606495
RSI: 36.7043546662207
MACD: -270.22342005127575
MACD Signal: -214.9665803482162
Volume Ratio: 0.2940486640748201


In [9]:
decision = atr_strategy.generate_entry_signal(
    current_price=current_price,
    atr=atr,
    rsi=rsi,
    macd=macd,
    macd_signal=macd_signal,
    volume_ratio=volume_ratio
)

print(decision)

{'action': 'HOLD', 'price': 77524.59, 'atr': 352.51520307606495, 'confirmations': 0, 'reason': 'Insufficient bullish confirmations'}
